# Lab Assignment 3 — Task 2

## Converting a CFG to Chomsky Normal Form (CNF)

Many parsing algorithms, including CKY, require the grammar to be in **Chomsky Normal Form (CNF)**.

A grammar is in CNF if its production rules have one of these forms:

1. `A -> B C` — a non-terminal produces two non-terminals.
2. `A -> 'word'` — a non-terminal produces a single terminal.

## PART 1 : The Challenge of Ambiguity & CNF Constraints

Consider:

> **I saw the man with a telescope.**

This sentence has two possible interpretations:

1. I saw a man who was holding a telescope.
2. I used a telescope to see the man.

A PCFG assigns probabilities to grammar rules. The probability of a complete parse tree is obtained by multiplying the probabilities of all production rules used in that tree.

### Your task

1. **Convert the given PCFG into strict CNF.** Identify the rule that violates CNF and replace it with equivalent binary rule(s). Preserve the probability of the original alternative.
2. **Parse the sentence using the CNF PCFG and `ViterbiParser`.** Print the most probable parse tree and its total probability.
3. **Explain mathematically why the parser selected this interpretation over the alternative.** Show the rule probabilities used in the selected tree and compare the resulting probability with the competing interpretation.

In [1]:
pcfg_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V NP PP [0.3]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

sentence = "I saw the man with a telescope"

In [2]:
import nltk
from nltk import PCFG
from nltk.parse import ViterbiParser

# Convert the non-CNF rule:
#     VP -> V NP PP [0.3]
# into binary rules. Choose an intermediate non-terminal and
# distribute the probability so that the original derivation
# keeps the same probability.

# Example:
#     VP -> V VP_PP [0.3]
#     VP_PP -> NP PP [1.0]

cnf_grammar_str = """
    S -> NP VP [1.0]
    NP -> 'I' [0.1] | Det N [0.3] | NP PP [0.6]
    VP -> V NP [0.7] | V VP_PP [0.3]
    VP_PP -> NP PP [1.0]
    PP -> P NP [1.0]
    Det -> 'the' [0.8] | 'a' [0.2]
    N -> 'man' [0.5] | 'telescope' [0.5]
    V -> 'saw' [1.0]
    P -> 'with' [1.0]
"""

### Analysis requirement

Do not hard-code the final probability.

Use the parse tree returned by the parser to determine which rules were selected. Then calculate the probability of the competing interpretation from the corresponding rule probabilities.

Your explanation should make clear that **ViterbiParser selects the parse with the highest product of rule probabilities**.

The rule `VP -> V NP PP [0.3]` is not in strict CNF because it has three symbols on the right-hand side. We replace it with two binary rules:

```text
VP -> V VP_PP [0.3]
VP_PP -> NP PP [1.0]
```

The original probability is preserved because `0.3 × 1.0 = 0.3`.

In [3]:
cnf_grammar = PCFG.fromstring(cnf_grammar_str)

parser = ViterbiParser(cnf_grammar)
tokens = sentence.split()

best_tree = next(parser.parse(tokens))

print("Most probable parse tree:")
print(best_tree)

print("\nProbability from parser:")
print(best_tree.prob())

Most probable parse tree:
(S
  (NP I)
  (VP
    (V saw)
    (NP
      (NP (Det the) (N man))
      (PP (P with) (NP (Det a) (N telescope)))))) (p=0.0001512)

Probability from parser:
0.0001512


### Explanation

The Viterbi parser chooses the parse tree with the highest product of the probabilities of the grammar rules used in that tree. The returned tree attaches `with a telescope` to `the man`.

In [4]:
def calculate_tree_probability(tree, grammar):
    probability = 1.0
    selected_rules = []

    for production in tree.productions():
        lhs = production.lhs()
        rhs = production.rhs()

        for grammar_production in grammar.productions(lhs):
            if grammar_production.rhs() == rhs:
                p = grammar_production.prob()
                probability *= p
                selected_rules.append((str(grammar_production), p))
                break

    return probability, selected_rules


selected_probability, selected_rules = calculate_tree_probability(
    best_tree, cnf_grammar
)

print("Rules used in the selected parse:")
for rule, p in selected_rules:
    print(f"{rule} -> {p}")

print("\nCalculated probability:", selected_probability)
print("Parser probability:", best_tree.prob())

Rules used in the selected parse:
S -> NP VP [1.0] -> 1.0
NP -> 'I' [0.1] -> 0.1
VP -> V NP [0.7] -> 0.7
V -> 'saw' [1.0] -> 1.0
NP -> NP PP [0.6] -> 0.6
NP -> Det N [0.3] -> 0.3
Det -> 'the' [0.8] -> 0.8
N -> 'man' [0.5] -> 0.5
PP -> P NP [1.0] -> 1.0
P -> 'with' [1.0] -> 1.0
NP -> Det N [0.3] -> 0.3
Det -> 'a' [0.2] -> 0.2
N -> 'telescope' [0.5] -> 0.5

Calculated probability: 0.0001512
Parser probability: 0.0001512


In [5]:
def get_rule_probability(grammar, lhs, rhs):
    for production in grammar.productions():
        production_rhs = tuple(
            symbol.symbol() if hasattr(symbol, "symbol") else symbol
            for symbol in production.rhs()
        )

        if production.lhs().symbol() == lhs and production_rhs == tuple(rhs):
            return production.prob()

    raise ValueError(f"Rule not found: {lhs} -> {rhs}")


competing_rules = [
    ("S", ["NP", "VP"]),
    ("NP", ["I"]),
    ("VP", ["V", "VP_PP"]),
    ("V", ["saw"]),
    ("VP_PP", ["NP", "PP"]),
    ("NP", ["Det", "N"]),
    ("Det", ["the"]),
    ("N", ["man"]),
    ("PP", ["P", "NP"]),
    ("P", ["with"]),
    ("NP", ["Det", "N"]),
    ("Det", ["a"]),
    ("N", ["telescope"])
]

competing_probability = 1.0

print("Rules used in competing interpretation:")

for lhs, rhs in competing_rules:
    p = get_rule_probability(cnf_grammar, lhs, rhs)
    competing_probability *= p
    print(f"{lhs} -> {' '.join(rhs)} -> {p}")

print("\nCompeting interpretation probability:", competing_probability)

Rules used in competing interpretation:
S -> NP VP -> 1.0
NP -> I -> 0.1
VP -> V VP_PP -> 0.3
V -> saw -> 1.0
VP_PP -> NP PP -> 1.0
NP -> Det N -> 0.3
Det -> the -> 0.8
N -> man -> 0.5
PP -> P NP -> 1.0
P -> with -> 1.0
NP -> Det N -> 0.3
Det -> a -> 0.2
N -> telescope -> 0.5

Competing interpretation probability: 0.00010800000000000001


## Mathematical Comparison

Selected interpretation:

```text
I saw [the man [with a telescope]]
```

Its probability is `0.0001512`.

Competing interpretation:

```text
I saw [the man] [with a telescope]
```

Its probability is `0.000108`.

Since `0.0001512 > 0.000108`, Viterbi selects the first interpretation. The ratio is `0.0001512 / 0.000108 = 1.4`, so the selected interpretation is 1.4 times more probable.

# Part 2: Advanced Dependency Parsing Using spaCy

Use the **spaCy NLP library** to perform dependency parsing and graph traversal on the following sentence:

> **The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.**


## 1. Initialize and Parse

Load the `en_core_web_sm` model in spaCy and process the sentence.

Return the parsed object and store it as **`parsed_doc`**.

In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")

sentence = (
    "The exhausted researchers at the institute discovered "
    "a new vaccine that completely prevents the viral mutation."
)

parsed_doc = nlp(sentence)

print(sentence)

The exhausted researchers at the institute discovered a new vaccine that completely prevents the viral mutation.


## 2. Isolate the Root

Create a function that accepts **`parsed_doc` as its only parameter**.

Iterate through the document to find the structural center of the sentence, where the dependency tag is `"ROOT"`.

Return this token and save it as **`root_node`**.


In [7]:
def find_root(parsed_doc):
    for token in parsed_doc:
        if token.dep_ == "ROOT":
            return token

    return None


root_node = find_root(parsed_doc)

print("Root word:", root_node.text)

Root word: discovered


## 3. Extract the Graph Terminals

Write a function that requires **both `root_node` and `parsed_doc`** as inputs.

1. Check the `.children` of `root_node` to find the nominal subject (`nsubj` dependency), isolating **`researchers`** as `start_node`.
2. Find the token whose text is **`mutation`** in `parsed_doc`, isolating it as `end_node`.
3. Return both nodes.

In [8]:
def find_graph_terminals(root_node, parsed_doc):
    start_node = None
    end_node = None

    for child in root_node.children:
        if child.dep_ == "nsubj":
            start_node = child
            break

    for token in parsed_doc:
        if token.text.lower() == "mutation":
            end_node = token
            break

    return start_node, end_node


start_node, end_node = find_graph_terminals(root_node, parsed_doc)

print("Start node:", start_node.text)
print("End node:", end_node.text)

Start node: researchers
End node: mutation


## 4. Graph Traversal — Shortest Path

Treat the dependency parse as an **undirected graph**. From any token, you may move to:

- its `.head`, or
- any of its `.children`.

Write a **Breadth-First Search (BFS)** function that takes `start_node` and `end_node` as inputs.

Find the shortest path between the two words and return it as a list of tokens named **`dependency_path`**.

In [9]:
from collections import deque


def bfs_shortest_path(start_node, end_node):
    queue = deque([(start_node, [start_node])])
    visited = {start_node.i}

    while queue:
        current, path = queue.popleft()

        if current == end_node:
            return path

        neighbors = [current.head] + list(current.children)

        for neighbor in neighbors:
            if neighbor.i not in visited:
                visited.add(neighbor.i)
                queue.append((neighbor, path + [neighbor]))

    return []


dependency_path = bfs_shortest_path(start_node, end_node)

print("Shortest dependency path:")
print(" -> ".join(token.text for token in dependency_path))

Shortest dependency path:
researchers -> discovered -> vaccine -> prevents -> mutation


## 5. Path Analysis

Write a function that accepts **`dependency_path`** as its input.

Iterate through **this list of tokens only**, not the whole sentence, and print their syntactic details in the following format:

| Word | POS | Head | Dependency |
|---|---|---|---|
| researchers | NOUN | discovered | nsubj |
| ... | ... | ... | ... |

This should extract only the most important syntactic skeleton connecting the two concepts.

In [10]:
def print_path_analysis(dependency_path):
    print(f"{'Word':<15}{'POS':<10}{'Head':<15}{'Dependency'}")
    print("-" * 55)

    for token in dependency_path:
        print(
            f"{token.text:<15}"
            f"{token.pos_:<10}"
            f"{token.head.text:<15}"
            f"{token.dep_}"
        )


print_path_analysis(dependency_path)

Word           POS       Head           Dependency
-------------------------------------------------------
researchers    NOUN      discovered     nsubj
discovered     VERB      discovered     ROOT
vaccine        NOUN      discovered     dobj
prevents       VERB      vaccine        relcl
mutation       NOUN      prevents       dobj


## 6. Visual Verification

Pass the original **`parsed_doc`** into spaCy's `displacy` module to render the full dependency tree.

Use the resulting diagram to visually verify that the table generated in Step 5 matches the shortest path in the dependency tree.

In [11]:
from spacy import displacy

displacy.render(
    parsed_doc,
    style="dep",
    jupyter=True
)